# ProxyTrace 05. ESG 태깅 + 판별 순서 정하기

04에서 뽑은 주장들에 **ESG 정보를 붙이고, 06에서 무엇부터 판별할지 줄을 세워요.**

GPT를 쓰지 않고 **사람이 정한 기준표**로 해요. 그래야 "왜 이 주장이 1순위인가?"를 분명하게 설명할 수 있어요.

| 붙이는 정보 | 뜻 |
| --- | --- |
| `esg_pillar` | E(환경) / S(사회) / G(지배구조) |
| `sasb_issue` | Meta 업종(인터넷 미디어)에서 중요한 ESG 이슈 이름 |
| `materiality` | 중대성: high / medium / low |
| `proponent_stance` | 제안자 성향: pro_esg / anti_esg / governance / unknown |
| `priority` | 판별 순서: 1순위 / 2순위 / 3순위 / 제외 |
| `priority_reason` | 왜 그 순위인지 (점수 내역) |

결과: `data/claims/{TICKER}/{제출일}_claims_esg.json`, `.csv`

## 1. 설정 + 04 결과 불러오기

In [18]:
import csv
import json
from collections import Counter
from pathlib import Path

# ▶ 01~04 노트북과 같은 티커
TICKER = "META"

T = TICKER.lower()
manifest = json.loads((Path("data/raw") / T / "manifest.json").read_text(encoding="utf-8"))
FILING = manifest["proxies"][0]["filingDate"]
CLAIM_DIR = Path("data/claims") / T

# 04에서 전체(10개)를 돌렸으면 그 파일을, 아니면 테스트 파일을 읽어요
src = CLAIM_DIR / f"{FILING}_claims.json"
if not src.exists():
    src = CLAIM_DIR / f"{FILING}_claims_test.json"
    print("⚠ 전체 결과가 없어서 테스트 결과로 진행해요 (04에서 TEST_NUMBERS = None으로 돌리면 전체가 생겨요)")
claims = json.loads(src.read_text(encoding="utf-8"))["claims"]
print(f"{src.name}: 주장 {len(claims)}개, 안건 {sorted({c['proposal'] for c in claims})}")

⚠ 전체 결과가 없어서 테스트 결과로 진행해요 (04에서 TEST_NUMBERS = None으로 돌리면 전체가 생겨요)
2026-04-16_claims_test.json: 주장 16개, 안건 [4]


중복 합치기 함수

In [19]:
# 회사 반론은 "요약 bullet → 자세한 설명" 순서라 같은 사실을 두 번 말해요.
# 같은 안건·같은 화자의 주장 중에서 '원문 단어'와 '한국어 요약'이 둘 다 많이 겹치면 같은 주장으로 봐요.
import re

STOP = set("a an the of to in on for and or our we is are was were be by at as that this with from its it".split())
QUOTE_TH, KO_TH = 0.6, 0.6     # 겹침 기준 (높이면 덜 합치고, 낮추면 더 합쳐요)


def words(s: str) -> set:
    return {w for w in re.findall(r"[a-z0-9\-]+", s.lower()) if w not in STOP}


def bigrams(s: str) -> set:
    s = re.sub(r"\s+", "", s)
    return {s[i:i + 2] for i in range(len(s) - 1)}


def overlap(a: set, b: set) -> float:
    return len(a & b) / min(len(a), len(b)) if a and b else 0.0


def same_claim(x: dict, y: dict) -> bool:
    if (x["proposal"], x["speaker"]) != (y["proposal"], y["speaker"]):
        return False
    return (overlap(words(x["quote"]), words(y["quote"])) >= QUOTE_TH
            and overlap(bigrams(x["claim_ko"]), bigrams(y["claim_ko"])) >= KO_TH)


def dedup(claims: list[dict]) -> list[dict]:
    """같은 주장끼리 묶고 대표 하나만 남겨요. 대표 = ★(회사 핵심 주장) > 먼저 나온 것"""
    groups = []
    for c in claims:
        for g in groups:
            if same_claim(g[0], c):
                g.append(c); break
        else:
            groups.append([c])
    out = []
    for g in groups:
        rep = sorted(g, key=lambda c: (not c.get("in_board_summary"), claims.index(c)))[0]
        rep = {**rep, "merged_ids": [c["claim_id"] for c in g if c is not rep],
               "pages_all": sorted({c["page_no"] for c in g if c["page_no"] is not None})}
        out.append(rep)
    return out

중복 합치기 실행

In [20]:
merged = dedup(claims)
print(f"주장 {len(claims)}개 → 중복 합친 뒤 {len(merged)}개")
for c in merged:
    if c["merged_ids"]:
        print(f"  {c['claim_id']} ← {c['merged_ids']} 합침 | {c['claim_ko'][:50]}")

주장 16개 → 중복 합친 뒤 15개
  P04-C02 ← ['P04-C07'] 합침 | 2025년 주주 총회에서 주주들은 '급여에 대한 의사 표시' 투표를 3년에 한 번 실시하는


## 2. ESG 매핑표 + 제안자 성향
⚠ 제안자 성향은 **초안**이에요. 기획서에 쓰기 전에 직접 확인하고 고치세요.

In [21]:
# ── ESG 매핑표 ────────────────────────────────────────────
# 04에서 GPT가 붙인 topic → E/S/G + SASB 이슈 + 중대성
# SASB '인터넷 미디어·서비스' 업종 기준. 중대성(materiality)은 직접 조정해도 돼요.
#   high   = Meta 업종에서 재무적으로 중요한 SASB 이슈
#   medium = 업종 이슈는 아니지만 투자자가 공통으로 중요하게 보는 이슈 (지배구조 등)
#   low    = 그 외
ESG_MAP = {
    "privacy_ai_data":       ("S", "Data Privacy & Data Security",                 "high"),
    "human_rights_content":  ("S", "Advertising Standards & Freedom of Expression", "high"),
    "climate_energy":        ("E", "Environmental Footprint of Hardware Infrastructure", "high"),
    "workforce_immigration": ("S", "Employee Recruitment, Inclusion & Performance", "high"),
    "child_safety":          ("S", "Advertising Standards & Freedom of Expression", "medium"),
    "voting_governance":     ("G", "(지배구조 공통 이슈)",                           "medium"),
    "compensation":          ("G", "(지배구조 공통 이슈)",                           "medium"),
    "other":                 ("-", "-",                                            "low"),
}

# ── 제안자 성향 ───────────────────────────────────────────
# ⚠ 초안이에요. 제안자 웹사이트·과거 제안 이력을 직접 확인하고 고치세요.
#   pro_esg   = ESG 공시·정책 강화를 요구하는 쪽
#   anti_esg  = ESG 정책에 반대하거나 문제 삼는 쪽
#   governance = 주주 권리·지배구조 개선이 중심
#   unknown   = 모름 (기본값)
STANCE = {
    "National Legal and Policy Center": "anti_esg",
    "National Center for Public Policy Research": "anti_esg",
    "John Chevedden": "governance",
    "NorthStar Asset Management, Inc. Funded Pension Plan": "governance",
    "Treasurer for the State of Illinois and Trustee of the Bright Directions College Savings Trust": "governance",
    "Azzad Asset Management, Inc": "pro_esg",
    "JLens on behalf of the Leichtag Foundation": "unknown",
    "As You Sow on behalf of As You Sow Foundation Fund": "pro_esg",
    "Proxy Impact on behalf of the Hyde Cragmont 2015 Trust": "pro_esg",
    "Mercy Investment Services, Inc": "pro_esg",
}

## 3. 우선순위 규칙
```
검증 불가 주장            → 제외
중대성 high / medium / low → +3 / +2 / +1
회사 핵심 주장(★)          → +2
수치 주장                  → +1

5점 이상 → 1순위 / 3~4점 → 2순위 / 그 아래 → 3순위
```

In [22]:
# ── 우선순위 점수 ─────────────────────────────────────────
# 점수가 높을수록 06에서 먼저, 깊게 검증해요. 기준을 바꾸고 싶으면 숫자만 고치면 돼요.
SCORE = {
    "materiality": {"high": 3, "medium": 2, "low": 1},
    "board_summary": 2,     # 회사가 요약 bullet에 직접 넣은 핵심 주장(★)
    "number": 1,            # 숫자가 핵심인 주장 (확인 결과가 분명하게 나옴)
}
TIERS = [(5, "1순위"), (3, "2순위"), (0, "3순위")]


def stance_of(name: str) -> str:
    for k, v in STANCE.items():
        if name and (k.lower() in name.lower() or name.lower() in k.lower()):
            return v
    return "unknown"


def tag(c: dict) -> dict:
    pillar, sasb, mat = ESG_MAP.get(c["topic"], ESG_MAP["other"])
    out = {**c, "esg_pillar": pillar, "sasb_issue": sasb, "materiality": mat,
           "proponent_stance": stance_of(c["proponent"])}

    # 의혹 제기는 판별은 못 해도 담당자가 꼭 봐야 해서 따로 표시해요
    out["flag"] = "의혹 제기" if c["kind"] == "allegation" else ""

    if not c["checkable"]:
        why = {"request": "요구", "opinion_commitment": "의견·약속", "allegation": "의혹 제기 (결과표에 별도 표시)"}
        out.update(priority_score=0, priority="제외", priority_reason=f"검증 불가 ({why.get(c['kind'], c['kind'])})")
        return out

    score = SCORE["materiality"][mat]
    reason = [f"중대성 {mat}(+{SCORE['materiality'][mat]})"]
    if c.get("in_board_summary"):
        score += SCORE["board_summary"]; reason.append(f"회사 핵심 주장★(+{SCORE['board_summary']})")
    if c["kind"] == "number":
        score += SCORE["number"]; reason.append(f"수치 주장(+{SCORE['number']})")

    tier = next(t for th, t in TIERS if score >= th)
    out.update(priority_score=score, priority=tier, priority_reason=" / ".join(reason))
    return out

## 4. 실행

In [23]:
tagged = [tag(c) for c in merged]
print("순위별:", dict(Counter(c["priority"] for c in tagged)))
print("ESG별 (검증 가능한 것만):", dict(Counter(c["esg_pillar"] for c in tagged if c["checkable"])))

순위별: {'제외': 6, '3순위': 8, '2순위': 1}
ESG별 (검증 가능한 것만): {'G': 8, '-': 1}


## 5. 결과 확인

In [ ]:
import pandas as pd

df = pd.DataFrame(tagged).reset_index(drop=True)
order = {"1순위": 0, "2순위": 1, "3순위": 2, "제외": 3}
df = df.sort_values(by=["priority", "priority_score"], key=lambda s: s.map(order) if s.name == "priority" else -s)
cols = ["priority", "priority_score", "claim_id", "merged_ids", "speaker", "kind", "esg_pillar",
        "materiality", "proponent_stance", "flag", "claim_ko", "priority_reason"]
pd.set_option("display.max_colwidth", 80)
df[cols].set_index("priority")  # 왼쪽 줄 번호 숨기기

AttributeError: The '.style' accessor requires jinja2

In [25]:
# 안건별 요약: 안건마다 몇 개를 판별해야 하는지
summary = (df[df["priority"] != "제외"]
           .groupby(["proposal", "proposal_topic"])["priority"]
           .value_counts().unstack(fill_value=0))
summary

,priority,2순위,3순위
proposal,proposal_topic,,
4,Annual Vote Regarding Executive Pay,1,8


## 6. 저장

In [ ]:
jpath = CLAIM_DIR / f"{FILING}_claims_esg.json"
jpath.write_text(json.dumps({"ticker": TICKER, "filingDate": FILING, "source": src.name,
                             "claims": tagged}, ensure_ascii=False, indent=2), encoding="utf-8")
cpath = CLAIM_DIR / f"{FILING}_claims_esg.csv"
with cpath.open("w", newline="", encoding="utf-8-sig") as f:
    w = csv.DictWriter(f, fieldnames=list(tagged[0].keys()))
    w.writeheader(); w.writerows(tagged)
print(f"저장: {jpath}\n저장: {cpath}")

저장: data/claims/meta/2026-04-16_claims_esg.json
저장: data/claims/meta/2026-04-16_claims_esg.csv


## 다음 단계
5번 셀 표를 보내 주세요. **1순위 주장들**이 06에서 에이전트가 가장 먼저 판별할 대상이에요.

순위가 이상하면 2번 셀(매핑표)이나 3번 셀(점수)의 숫자를 고치면 돼요.